## Paso 1 — Cargamos el dataset ya tipado

Leemos directamente el Parquet generado en `00_setup_y_carga.ipynb`, sin
depender de que ese notebook se haya corrido antes en esta misma sesión.
Esto es lo único que este notebook necesita del anterior.


In [39]:
import pandas as pd
from pathlib import Path

INTERIM_DIR = Path("../datos/interim")
df = pd.read_parquet(INTERIM_DIR / "viajes_btura_tipado.parquet")

print(df.shape)
df.dtypes


(41237, 17)


idClaseDocumentoFwo                 str
fo                                Int64
fechaCreacionFo          datetime64[us]
bpCliente                         Int64
origen                              str
destino                             str
distanciaTotal                  float64
pesoNeto                        float64
posicionPrincipalFwop               str
grupoDeEquipoFwop                   str
vehiculo                            str
idMedioTransporteFo                 str
claseEquipoFop                      str
propiedad                           str
tipo                                str
tarifaVentaAlCliente            float64
fleteProduccionFo               float64
dtype: object

## Paso 2 — Dimensiones y peso en memoria

Filas, columnas y memoria RAM que ocupa el DataFrame ya cargado.

In [40]:
n_filas, n_columnas = df.shape
memoria_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)

print(f"Filas:      {n_filas:,}")
print(f"Columnas:   {n_columnas}")
print(f"Memoria:    {memoria_mb:.2f} MB")


Filas:      41,237
Columnas:   17
Memoria:    7.88 MB


## Paso 3 — Tabla resumen por columna

Tipo, nulos, % de nulos, valores únicos, cardinalidad relativa y un ejemplo
por columna. Tipo declarado e inferido coinciden porque venimos de un
Parquet con schema explícito (fijado en el notebook 00) — no hay inferencia
de por medio.


In [41]:
resumen = pd.DataFrame({
    "columna": df.columns,
    "tipo": [str(df[c].dtype) for c in df.columns],
    "nulos": [df[c].isna().sum() for c in df.columns],
    "unicos": [df[c].nunique() for c in df.columns],
})
resumen["pct_nulos"] = (resumen["nulos"] / len(df) * 100).round(2)
resumen["cardinalidad_relativa"] = (resumen["unicos"] / len(df)).round(4)
resumen["ejemplo"] = [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns]

resumen = resumen[["columna", "tipo", "nulos", "pct_nulos", "unicos", "cardinalidad_relativa", "ejemplo"]]
resumen


,columna,tipo,nulos,pct_nulos,unicos,cardinalidad_relativa,ejemplo
0,idClaseDocumentoFwo,str,0,0.00,7,0.0002,OPIM
1,fo,Int64,0,0.00,41107,0.9968,301403469
2,fechaCreacionFo,datetime64[us],0,0.00,371,0.0090,2026-03-31 00:00:00
3,bpCliente,Int64,0,0.00,537,0.0130,1
4,origen,str,0,0.00,113,0.0027,BUENAVENTURA
5,destino,str,0,0.00,122,0.0030,GIRON
6,distanciaTotal,float64,0,0.00,287,0.0070,712.0
7,pesoNeto,float64,0,0.00,8117,0.1968,7970.0
8,posicionPrincipalFwop,str,0,0.00,3,0.0001,TUR
9,grupoDeEquipoFwop,str,7496,18.18,2,0.0000,CN


## Paso 4 — head(), tail() y muestra aleatoria

Las primeras filas no bastan para conocer el dato: pueden estar ordenadas y
dar una idea sesgada. Revisamos también el final del archivo (donde sabemos
que hay un mes parcial) y una muestra aleatoria con semilla fija.


In [42]:
df.head()

,idClaseDocumentoFwo,fo,fechaCreacionFo,bpCliente,origen,destino,distanciaTotal,pesoNeto,posicionPrincipalFwop,grupoDeEquipoFwop,vehiculo,idMedioTransporteFo,claseEquipoFop,propiedad,tipo,tarifaVentaAlCliente,fleteProduccionFo
0,OPIM,301403469,2026-03-31,1,BUENAVENTURA,GIRON,712.0,7970.0,TUR,CN,TTX595,TC-2S2,CNT40.HC,Tercero,TC.2S2,5800000.0,5800000.0
1,OPTN,301269707,2025-07-31,100229,BUENAVENTURA,PUERTO GAITAN,826.0,34000.0,TUR,CAM,SSQ288,TC-3S3,3S3.34T,Tercero,TC.3S3,9860000.0,8874000.0
2,OPTN,301272820,2025-08-06,100229,BUENAVENTURA,PUERTO GAITAN,826.0,34000.0,TUR,CAM,SXV119,TC-3S3,3S3.34T,Tercero,TC.3S3,10200000.0,9180000.0
3,OPTN,301276538,2025-08-14,100229,BUENAVENTURA,TURBO,788.0,34000.0,TUR,CAM,SWK320,TC-3S3,3S3.34T,Tercero,TC.3S3,8537500.0,8000000.0
4,OPTN,301277447,2025-08-15,100229,BUENAVENTURA,TURBO,788.0,9800.0,TUR,CAM,WEB025,TC_SENCI,C2.8T,Tercero,C2.8T,3400000.0,3400000.0


In [43]:
df.tail()

,idClaseDocumentoFwo,fo,fechaCreacionFo,bpCliente,origen,destino,distanciaTotal,pesoNeto,posicionPrincipalFwop,grupoDeEquipoFwop,vehiculo,idMedioTransporteFo,claseEquipoFop,propiedad,tipo,tarifaVentaAlCliente,fleteProduccionFo
41232,OPIM,301277023,2025-08-14,99089,MOSQUERA,BUENAVENTURA,514.0,4000.0,TUR,CN,SKG983,TC-2S3,CNT40.HC,Tercero,TC.2S3,1450000.0,1200000.0
41233,OPIM,301276835,2025-08-14,99089,FUNZA,BUENAVENTURA,521.0,4000.0,TUR,CN,SRO101,TC-2S2,CNT40.HC,Tercero,TC.2S2,1450000.0,1200000.0
41234,OPIM,301276945,2025-08-14,99089,FUNZA,BUENAVENTURA,521.0,4000.0,TUR,CN,UPQ521,TC-2S2,CNT40.HC,Tercero,TC.2S2,1450000.0,1200000.0
41235,OPIM,301276989,2025-08-14,99089,MOSQUERA,BUENAVENTURA,514.0,4000.0,TUR,CN,TRJ278,TC-2S2,CNT40.HC,Tercero,TC.2S2,1450000.0,1200000.0
41236,OPIM,301276900,2025-08-14,99089,MOSQUERA,BUENAVENTURA,514.0,4000.0,TUR,CN,SRL476,TC-2S2,CNT40.HC,Tercero,TC.2S2,1450000.0,1200000.0


In [44]:
df.sample(20, random_state=42)

,idClaseDocumentoFwo,fo,fechaCreacionFo,bpCliente,origen,destino,distanciaTotal,pesoNeto,posicionPrincipalFwop,grupoDeEquipoFwop,vehiculo,idMedioTransporteFo,claseEquipoFop,propiedad,tipo,tarifaVentaAlCliente,fleteProduccionFo
10874,OPIM,301221314,2025-05-06,1958,BUENAVENTURA,BUENAVENTURA,10.0,4000.0,TUR,CN,TMP785,TC-2S2,CNT40.HC,Tercero,TC.2S2,700000.0,730000.0
25501,OPIM,301211987,2025-04-15,74037,BUENAVENTURA,VILLA RICA,183.0,15480.0,TUR,CN,SQL145,TC-3S3,CNT40.HC,Tercero,TC.3S3,2440000.0,2100000.0
3253,OHUB,301337045,2025-11-27,12503,BUENAVENTURA,BUENAVENTURA,10.0,1000.0,TUR,CN,WBF168,TC-3S3,CNT20.HC,Tercero,TC.3S3,363098.0,310000.0
18092,OPIM,301232126,2025-05-26,580,BUENAVENTURA,MADRID,528.0,26350.0,TUR,CAM,SML871,TC-3S3,3S3.34T,Tercero,TC.3S3,5193746.0,5200000.0
30589,OPIM,301394144,2026-03-14,97676,BUENAVENTURA,DOSQUEBRADAS,297.0,34000.0,PRD,NaN,WLC184,TC-3S3,NaN,Tercero,TC.3S3,3230000.0,3230000.0
4508,OHUB,301286860,2025-09-01,14193,BUENAVENTURA,BUENAVENTURA,10.0,1000.0,TUR,CN,TRB115,TC-3S3,CNT20.ST,Tercero,TC.3S3,460000.0,460000.0
5488,OPIM,301404353,2026-04-03,14254,BUENAVENTURA,CALI,156.0,34000.0,PRD,NaN,SND794,TC-3S3,NaN,Tercero,TC.3S3,2720000.0,2720000.0
1392,OPIM,301223360,2025-05-10,107936,BUENAVENTURA,GUARNE,498.0,32260.0,TUR,CAM,TRN099,TC-3S3,3S3.34T,Tercero,TC.3S3,5600000.0,5600000.0
1670,OPIM,301268961,2025-07-30,110592,BUENAVENTURA,MOSQUERA,503.0,26610.0,TUR,CN,SVF178,TC-3S3,CNT40.HC,Tercero,TC.3S3,6200000.0,6200000.0
26547,OPIM,301210328,2025-04-11,761,BUENAVENTURA,TOCANCIPA,498.0,23100.0,TUR,CN,SOT092,TC-3S2,CNT40.HC,Tercero,TC.3S2,5500000.0,5500000.0


## Paso 5 — Clasificación de columnas por naturaleza

| Columna | Clasificación | Nota |
|---|---|---|
| `fo` | Identificador | No es 100% único — 130 valores repetidos (a resolver en la fase de unicidad) |
| `fechaCreacionFo` | Fecha | Administrativa (fecha de creación en sistema, no de ejecución del viaje) |
| `bpCliente` | Identificador (código) | Sin maestro de nombres disponible |
| `idClaseDocumentoFwo` | Categórica nominal | 7 valores presentes de 10 posibles en el catálogo CLASE FWO |
| `origen` / `destino` | Categórica nominal (alta cardinalidad) | Nombres de población; candidatas a normalización (ver muestra del Paso 4) |
| `distanciaTotal` | Numérica continua | En KM según diccionario |
| `pesoNeto` | Numérica continua | Unidad no confirmada (kg vs t) |
| `posicionPrincipalFwop` | Categórica jerárquica (nivel 1) | PKG / PRD / TUR — determina si aplican las dos columnas de equipo |
| `grupoDeEquipoFwop` | Categórica jerárquica (nivel 2) | Solo aplica bajo TUR |
| `claseEquipoFop` | Categórica jerárquica (nivel 3) | Solo aplica bajo CN y CAM |
| `vehiculo` | Identificador | Placa del vehículo |
| `idMedioTransporteFo` | Categórica nominal | Incluye un valor basura (`"1"`) |
| `propiedad` | Categórica nominal binaria | Tercero / Propio |
| `tipo` | Categórica nominal (descartable) | El diccionario la define como "puede omitirse" |
| `tarifaVentaAlCliente` | Numérica monetaria | Moneda e IVA sin confirmar |
| `fleteProduccionFo` | Numérica monetaria | Moneda e IVA sin confirmar |


## Paso 6 — Columnas constantes o cuasi-constantes

Medimos, por columna, qué porcentaje de las filas comparte el valor más
frecuente. Una concentración muy alta (o un solo valor único) señala una
columna sin poder informativo para el análisis.

In [45]:
resultados = []
for col in df.columns:
    frecuencias = df[col].value_counts(dropna=False, normalize=True)
    resultados.append({
        "columna": col,
        "valor_dominante": frecuencias.index[0],
        "concentracion": round(frecuencias.iloc[0], 4),
        "valores_unicos": df[col].nunique(dropna=False),
    })

concentracion_df = pd.DataFrame(resultados).sort_values("concentracion", ascending=False)
concentracion_df


,columna,valor_dominante,concentracion,valores_unicos
13,propiedad,Tercero,0.9117,2
8,posicionPrincipalFwop,TUR,0.8182,3
4,origen,BUENAVENTURA,0.7474,113
9,grupoDeEquipoFwop,CN,0.6876,3
0,idClaseDocumentoFwo,OPIM,0.6745,7
11,idMedioTransporteFo,TC-3S3,0.5417,15
14,tipo,TC.3S3,0.5375,18
12,claseEquipoFop,CNT40.HC,0.4711,28
5,destino,BUENAVENTURA,0.4700,122
6,distanciaTotal,10.0,0.2175,287


## Síntesis del notebook 01 — Perfilado estructural

**Tamaño:** 41.237 filas × 17 columnas, 7.88 MB en memoria — dataset liviano,
sin restricciones de cómputo para el resto del EDA.

**Tipos:** los 17 tipos fijados en `00_setup_y_carga.ipynb` (Paso 0-6) se
confirman consistentes al recargar desde Parquet — no hay divergencia entre
tipo declarado e inferido, señal de que el tipado quedó bien hecho.

**Nulos:** concentrados en 2 columnas (`grupoDeEquipoFwop` 18,18%,
`claseEquipoFop` 18,19%), coincidiendo exactamente con el baseline del plan
(Anexo A) — estructurales según el diccionario, no defecto. El veredicto de
calidad formal se confirma en `02_calidad_del_dato.ipynb`.

**Unicidad:** `fo` tiene 41.107 valores únicos sobre 41.237 filas (130
repetidos) — el grano de la fila sigue sin resolver. Se investiga en el
notebook `02` (Fase 3 del plan, §4).

**Orden del archivo:** confirmado que las filas NO están ordenadas
cronológicamente (el `tail()` mostró fechas de agosto 2025, no de la punta
de la ventana). Implicación: cualquier análisis temporal debe ordenar
explícitamente por fecha, nunca asumir el orden del archivo.

**Columnas constantes/cuasi-constantes:** ninguna. La más concentrada
(`propiedad`, 91,17% Tercero) todavía deja una categoría minoritaria con
volumen suficiente para análisis — se marca como posible sesgo de
representación, no como columna a descartar (ver Fase 10, §11).

**Clasificación de columnas:** completada (identificador / fecha /
categórica nominal / categórica jerárquica / numérica continua / numérica
monetaria) — ver Paso 5. Sienta las reglas de qué análisis aplica a cada
columna en los notebooks 03 y 04.

**Hallazgo lateral:** variante de nombre de ciudad detectada en la muestra
(`"BOGOTA, DISTRITO CAPITAL"`) — candidata a normalización en `02`.

**Pendientes para notebooks siguientes:**
- Cruce `bpCliente` ↔ maestro de clientes (acceso disponible) — `02`, §5.6.
- Cruce `origen`/`destino` ↔ DIVIPOLA del DANE (insumo externo por
  conseguir) — `02` y `05`.
- Perfilado automático (`ydata-profiling`/`sweetviz`) — **omitido**: ya
  cubierto manualmente sin riesgo de conflicto de versiones con
  pandas 3.0.5 / numpy 2.5.2.
